In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import timm
from tqdm import tqdm

/home/kgiri/ViT-Research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_ds = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=train_transform
)

test_ds = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=test_transform
)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)


100%|██████████| 169M/169M [00:03<00:00, 43.8MB/s] 


In [ ]:
model = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=False,        # FROM SCRATCH
    num_classes=100          # CIFAR-100
)
model = model.to(device)


In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20  # number of epochs
)


In [7]:
scaler = torch.cuda.amp.GradScaler()


/tmp/ipykernel_436148/2655216266.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [8]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0
    pbar = tqdm(train_dl, desc=f"Epoch {epoch+1} Training")

    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        pbar.set_postfix({"loss": total_loss / (len(pbar) + 1)})

    scheduler.step()
    return total_loss / len(train_dl)


In [9]:
def validate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in test_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total


In [11]:
epochs = 100
for epoch in range(epochs):
    loss = train_one_epoch(epoch)
    acc = validate()
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={acc:.2f}%")


Epoch 1 Training:   0%|          | 0/391 [00:00<?, ?it/s]/tmp/ipykernel_436148/2506865690.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1 Training: 100%|██████████| 391/391 [00:16<00:00, 23.36it/s, loss=3.62] 


Epoch 1: Loss=3.6309, Accuracy=17.37%


Epoch 2 Training: 100%|██████████| 391/391 [00:15<00:00, 25.98it/s, loss=3.26] 


Epoch 2: Loss=3.2656, Accuracy=23.40%


Epoch 3 Training: 100%|██████████| 391/391 [00:17<00:00, 22.98it/s, loss=2.98] 


...

Epoch 5: Loss=2.5776, Accuracy=34.34%


Epoch 6 Training: 100%|██████████| 391/391 [00:16<00:00, 23.21it/s, loss=2.41] 


Epoch 6: Loss=2.4123, Accuracy=35.86%


Epoch 7 Training:  36%|███▌      | 140/391 [00:06<00:10, 23.30it/s, loss=0.807]


KeyboardInterrupt: 